# easymlops TS 端到端示例

面板(多实体)时序数据的完整流程: **防泄漏切分 → 分组特征工程 → Parallel 并行分支 → 分组 ARIMA 预测 → 持久化与单条推理**。

> 运行环境: `pip install -e .` 后可直接运行; 统计模型依赖 `statsmodels`, `GarchRegression` 依赖 `arch`。

In [1]:
import os, sys
try:
    import easymlops
except ImportError:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
    import easymlops

import numpy as np
import pandas as pd
print("easymlops:", easymlops.__file__)

easymlops: d:/datas/share/traes/easymlops-trae\easymlops\__init__.py


## 1. 面板数据与防泄漏切分

两个实体(A/B)各 90 天; `split_time_series` 保证同一实体的样本不跨越 train/test 时间边界。

In [2]:
from easymlops.ts.utils import split_time_series

rng = np.random.RandomState(11)
n = 90
dates = pd.date_range("2026-01-01", periods=n, freq="D")
rows = []
for g, base in [("A", 50.0), ("B", 120.0)]:
    vals = base + np.cumsum(rng.normal(0, 1, n))          # 随机游走, 各组起点不同
    for t, v in zip(dates, vals):
        rows.append({"group": g, "date": t, "value": v, "value2": v + rng.normal(0, 1)})
data = pd.DataFrame(rows)

train, test = split_time_series(data, time_col="date", group_cols=["group"], test_size=0.3)
test = test.sort_values(["group", "date"]).reset_index(drop=True)
print("train:", train.shape, "test:", test.shape)
print("每组测试行数:", test.groupby("group").size().to_dict())

train: (124, 4) test: (56, 4)
每组测试行数: {'A': 28, 'B': 28}


## 2. 分组特征管道

所有特征 pipe 支持 `group_cols`(面板分组) 与 `time_col`(组内时间排序), 批量 transform 为纯函数, 单条 `transform_single` 流式演进且口径一致。

In [3]:
from easymlops import TSPipeLine
from easymlops.ts.features import RollingFeature, EWMAFeature, FourierFeature

pipe = TSPipeLine()
pipe.pipe(RollingFeature(cols=["value"], window=5, stats=["mean", "std"],
                         group_cols=["group"], time_col="date")) \
     .pipe(EWMAFeature(cols=["value"], span=7, group_cols=["group"], time_col="date")) \
     .pipe(FourierFeature(period=7, order=1, group_cols=["group"], time_col="date"))
pipe.fit(train)

out = pipe.transform(test)
feat_cols = [c for c in out.columns if c.startswith("value_") or c.startswith("fourier_")]
print("新增特征列:", feat_cols)
out[["group", "date"] + feat_cols].head()

the <class 'easymlops.ts.features.RollingFeature'> and <class 'easymlops.ts.core.group_pipe.GroupM2MRollingPipeBase'> use same parameter names [['min_periods', 'fillna_strategy', '_specs', 'group_cols', 'sort_rows', 'time_col']],please check if conflict,
the <class 'easymlops.ts.features.EWMAFeature'> and <class 'easymlops.ts.core.group_pipe.GroupM2MRollingPipeBase'> use same parameter names [['min_periods', 'fillna_strategy', '_specs', 'group_cols', 'sort_rows', 'time_col']],please check if conflict,
the <class 'easymlops.ts.features.FourierFeature'> and <class 'easymlops.ts.core.group_pipe.GroupTSBase'> use same parameter names [['time_col']],please check if conflict,
新增特征列: ['value_roll5_mean', 'value_roll5_std', 'value_ewma', 'value_roll5_mean_ewma', 'value_roll5_std_ewma', 'fourier_1_sin', 'fourier_1_cos']


## 3. Parallel 并行分支

`Parallel`(复用表模块) 让多个 pipe 接收相同输入并行计算后按列合并, 常用于同源数据派生多路特征。

In [4]:
from easymlops.table.ensemble import Parallel
from easymlops.ts.features import LagFeature

par = TSPipeLine()
par.pipe(Parallel(pipe_objects=[
    RollingFeature(cols=["value"], window=5, stats=["mean"],
                   group_cols=["group"], time_col="date"),
    LagFeature(cols=["value"], lags=[1], group_cols=["group"], time_col="date"),
])) \
  .pipe(FourierFeature(period=7, order=1, group_cols=["group"], time_col="date"))
par.fit(train)

pout = par.transform(test)
print("并行分支输出列:", [c for c in pout.columns if c not in ("group", "date", "value", "value2")])
print("结构:")
par.print_structs()

the <class 'easymlops.ts.features.LagFeature'> and <class 'easymlops.ts.core.group_pipe.GroupM2MRollingPipeBase'> use same parameter names [['min_periods', 'fillna_strategy', '_specs', 'group_cols', 'sort_rows', 'time_col']],please check if conflict,
并行分支输出列: ['value_roll5_mean', 'value2_roll5_mean', 'value_lag1', 'value2_lag1', 'fourier_1_sin', 'fourier_1_cos']
结构:
easymlops.table.ensemble.Parallel
|--easymlops.ts.features.RollingFeature
|--easymlops.ts.features.LagFeature
easymlops.ts.features.FourierFeature


## 4. 分组 ARIMA 预测

`ArimaRegression` 家族支持 `group_cols`: 每个实体独立拟合一个模型, forecast 延续各自序列; `transform_single` 输出该组下一步预测。

In [5]:
from easymlops.ts.statistical import ArimaRegression

arima = ArimaRegression(input_cols=["value"], p=1, d=1, q=1,
                        group_cols=["group"], time_col="date")
arima.fit(train)

pred = arima.transform(test)
summary = pred.groupby("group")["arima_predict"].agg(["first", "last"]).round(3)
summary.columns = ["首步预测", "末步预测"]
print(summary)

# 单条推理: 以 fit 序列为历史起点, 输出该组下一步预测
record = dict(test.iloc[0])
print("单条预测:", round(arima.transform_single(record)["arima_predict"], 3))
print("未见新组会明确报错: group ('C',) not fitted")

          首步预测     末步预测
group                  
A       49.405   49.374
B      109.293  109.478
单条预测: 49.405
未见新组会明确报错: group ('C',) not fitted


## 5. 持久化与部署

预测模型的拟合结果随 `save/load` 持久化(统计模型存模型对象, torch 深度模型存 state_dict), load 后无需重新 fit。

In [6]:
pkl = "_demo_arima.pkl"
arima.save(pkl)

loaded = ArimaRegression(input_cols=["value"], p=1, d=1, q=1,
                         group_cols=["group"], time_col="date")
loaded.load(pkl)

before = arima.transform(test)["arima_predict"].to_numpy()
after = loaded.transform(test)["arima_predict"].to_numpy()
print("load 后预测一致:", np.allclose(before, after))
os.remove(pkl)

load 后预测一致: True


## 6. 测试门禁

仓库自带全量门禁, 任一套件失败退出码为 1, 可直接接入 CI:

```bash
python tests/test_ts_all.py
```

覆盖: P0 基础特征 / P1 重采样与平稳性 / P2 变化量与幂变换 / P3 统计模型 /
P4 深度模型(权重持久化+单条推理) / statistical / deep_learning /
transformer / state_space / generative。